# 🚀 ML Augmentation — LightGBM + Stacking Ensemble

This notebook shows how we **augment** the probabilistic CLV baseline with gradient-boosted features, then combine both via a **Ridge meta-learner** to achieve the best predictions.

**Key result**: Stacking cuts MAE by 43.6% over BG/NBD alone.

In [ ]:
import sys
sys.path.insert(0, 'd:/ML_Project')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from src.config import *

import warnings
warnings.filterwarnings('ignore')

## 1. Model Comparison Results

Three models are compared on the holdout period:

In [ ]:
comparison = pd.read_parquet(OUTPUT_DATA_DIR / 'model_comparison.parquet')
comparison.style.format({'MAE': '{:,.0f}', 'RMSE': '{:,.0f}', 'MAPE': '{:.1f}%', 'Pearson_r': '{:.4f}'})

In [ ]:
# Bar chart comparing models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800']

comparison['MAE'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('MAE Comparison (Lower is Better)', fontsize=13)
axes[0].set_ylabel('MAE (INR)')
axes[0].tick_params(axis='x', rotation=0)

comparison['Pearson_r'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Pearson Correlation (Higher is Better)', fontsize=13)
axes[1].set_ylabel('Pearson r')
axes[1].set_ylim(0.85, 1.0)
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Model Performance Comparison', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

## 2. SHAP Feature Importance

SHAP (SHapley Additive exPlanations) reveals which features drive individual predictions.

In [ ]:
shap_data = joblib.load(MODELS_DIR / 'shap_values.pkl')
shap_values = shap_data['shap_values']
feature_names = shap_data['feature_names']
shap_values.feature_names = feature_names

print(f'SHAP values computed for {len(shap_values)} customers across {len(feature_names)} features')

In [ ]:
# Beeswarm plot showing feature importance + direction
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP Feature Importance (Beeswarm)', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Waterfall for a single high-value customer
shap.plots.waterfall(shap_values[0], show=False)
plt.title('SHAP Waterfall — Single Customer Explanation', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Predicted vs Actual (Stacked Model)

In [ ]:
clv_preds = pd.read_parquet(OUTPUT_DATA_DIR / 'clv_predictions.parquet')
holdout = pd.read_parquet(OUTPUT_DATA_DIR / 'holdout_actuals.parquet')

valid_idx = clv_preds.index.intersection(holdout.index)
y_true = holdout.loc[valid_idx, 'holdout_revenue'].fillna(0)
y_pred = clv_preds.loc[valid_idx, 'predicted_clv']

plt.figure(figsize=(8, 8))
plt.scatter(y_pred, y_true, alpha=0.3, s=10, color='green')
max_val = max(y_pred.quantile(0.99), y_true.quantile(0.99))
plt.plot([0, max_val], [0, max_val], 'r--', lw=2)
plt.xlabel('Stacked Predicted CLV (INR)', fontsize=12)
plt.ylabel('Actual Holdout Revenue (INR)', fontsize=12)
plt.title('Stacked Ensemble: Predicted vs Actual (r=0.990)', fontsize=14)
plt.xlim(0, max_val)
plt.ylim(0, max_val)
plt.tight_layout()
plt.show()

## 4. Residual Analysis

In [ ]:
residuals = y_true - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals.clip(residuals.quantile(0.01), residuals.quantile(0.99)),
             bins=50, color='purple', alpha=0.7, edgecolor='white')
axes[0].set_title('Residuals Distribution', fontsize=13)
axes[0].set_xlabel('Residual (Actual - Predicted)')
axes[0].axvline(0, color='red', linestyle='--')

axes[1].scatter(y_pred, residuals, alpha=0.3, s=10, color='purple')
axes[1].set_xlim(0, y_pred.quantile(0.99))
axes[1].set_ylim(residuals.quantile(0.01), residuals.quantile(0.99))
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted', fontsize=13)
axes[1].set_xlabel('Predicted CLV')
axes[1].set_ylabel('Residual')

plt.tight_layout()
plt.show()

## 5. Interpreting the Meta-Learner

Interestingly, the Ridge Stacker assigns a **negative coefficient (-0.13)** to the BG/NBD predictions, and a positive coefficient (1.09) to LightGBM.

**What does this mean?**
It suggests that LightGBM is highly accurate but tends to slightly over-predict for certain customers. The meta-learner uses the BG/NBD prediction as a *corrective downward signal*. This is a sophisticated interaction where the mathematical constraints of the probabilistic model help reign in the machine learning model's exuberance.

## Summary

| Model | MAE (INR) | Pearson r |
|-------|-----------|----------|
| BG/NBD | 1,04,836 | 0.8947 |
| LightGBM | 60,050 | 0.9898 |
| **Stacked** | **59,138** | **0.9902** |

- **Stacking reduces MAE by 43.6%** over the probabilistic baseline
- Ridge meta-learner coefficients (bgf=-0.13, lgbm=1.09) show it weights LightGBM heavily while using BG/NBD as a corrective signal
- SHAP reveals `monetary_value`, `frequency`, and `total_revenue` as the top 3 most important features